In [1]:
import os
import shutil
from datetime import date
from pathlib import Path
from dotenv import load_dotenv
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip


In [2]:
load_dotenv()

is_docker = os.path.exists("/.dockerenv")
MINIO_HOST = "minio:9000" if is_docker else "localhost:9000"
MINIO_USER = os.getenv("MINIO_ROOT_USER",     "minioadmin")
MINIO_PASS = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")

builder = (
    SparkSession.builder
    .appName("silver-fatos")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.defaultFS", "file:///")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

cliente_minio = Minio(
    MINIO_HOST, access_key=MINIO_USER, secret_key=MINIO_PASS, secure=False
)


:: loading settings :: url = jar:file:/mnt/c/Users/Laura/Documents/Disciplinas_SATC/engenharia_dados/projeto-ed-final/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/lauras/.ivy2.5.2/cache
The jars for the packages stored in: /home/lauras/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b7b98df2-9a95-4172-bebe-29ea2e3fee56;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.1.0 in central
	found io.delta#delta-storage;4.1.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
:: resolution report :: re

In [3]:
def download_delta_do_minio(bucket: str, prefix: str,
                            cliente: Minio, local_dir: Path) -> None:
    """Baixa toda a estrutura Delta (parquet + _delta_log) do MinIO para local."""
    if local_dir.exists():
        shutil.rmtree(local_dir)
    local_dir.mkdir(parents=True)
    objetos = list(cliente.list_objects(bucket, prefix=f"{prefix}/", recursive=True))
    print(f"  {len(objetos)} arquivo(s) em {bucket}/{prefix}/")
    for obj in objetos:
        relative = obj.object_name[len(prefix) + 1:]
        destino = local_dir / relative
        destino.parent.mkdir(parents=True, exist_ok=True)
        cliente.fget_object(bucket, obj.object_name, str(destino))


def upload_delta_para_minio(local_dir: Path, bucket: str, prefix: str,
                            cliente: Minio) -> None:
    """Sobe recursivamente a estrutura Delta para o MinIO."""
    for arq in sorted(local_dir.rglob("*")):
        if arq.is_file():
            object_name = f"{prefix}/{arq.relative_to(local_dir)}"
            cliente.fput_object(bucket, object_name, str(arq))
            print(f"  → {object_name}")


In [4]:
buckets = {b.name for b in cliente_minio.list_buckets()}
print("Buckets:", buckets)
if "silver" not in buckets:
    cliente_minio.make_bucket("silver")
    print("Bucket 'silver' criado.")


Buckets: {'landing', 'bronze'}
Bucket 'silver' criado.


In [5]:
local_musicas = Path("/tmp/bronze_local/musicas")
download_delta_do_minio("bronze", "dimensoes/musicas", cliente_minio, local_musicas)

df_musicas = (
    spark.read.format("delta").load(str(local_musicas))
    .select("id", "duracao_ms")
    .withColumnRenamed("id", "musica_id")
)
print(f"Musicas carregadas: {df_musicas.count():,}")


  6 arquivo(s) em bronze/dimensoes/musicas/


26/06/21 16:02:17 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 2:======================================================>  (48 + 2) / 50]

Musicas carregadas: 10,000


In [6]:
print("\n" + "=" * 55)
print("Silver: reproducoes")
print("=" * 55)

local_bronze = Path("/tmp/bronze_local/reproducoes")
download_delta_do_minio("bronze", "fatos/reproducoes", cliente_minio, local_bronze)
df = spark.read.format("delta").load(str(local_bronze))
print(f"Bronze lido: {df.count():,} registros")

df = df.dropDuplicates(["id"])

antes = df.count()
df = df.filter(F.col("ms_tocados") >= 30000)
depois = df.count()
print(f"Plays inválidos removidos: {antes - depois:,} | Válidos: {depois:,}")

df = (
    df.join(df_musicas, on="musica_id", how="left")
      .withColumn("completou", F.col("ms_tocados") >= (F.col("duracao_ms") * 0.9))
      .drop("duracao_ms")
)

df = df.withColumn("ano_mes", F.date_format(F.col("timestamp"), "yyyy-MM"))

local_silver = Path("/tmp/silver/reproducoes")
if local_silver.exists():
    shutil.rmtree(local_silver)

df.write.format("delta").mode("overwrite").partitionBy("ano_mes").save(str(local_silver))
print(f"Silver Delta escrito em {local_silver}")

print("Enviando para MinIO silver/fatos/reproducoes/")
upload_delta_para_minio(local_silver, "silver", "fatos/reproducoes", cliente_minio)
print(f"reproducoes Silver concluído — {depois:,} plays válidos.")



Silver: reproducoes
  8 arquivo(s) em bronze/fatos/reproducoes/
Bronze lido: 70,000 registros


Plays inválidos removidos: 13,991 | Válidos: 56,009


Silver Delta escrito em /tmp/silver/reproducoes
Enviando para MinIO silver/fatos/reproducoes/
  → fatos/reproducoes/_delta_log/.00000000000000000000.crc.crc
  → fatos/reproducoes/_delta_log/.00000000000000000000.json.crc
  → fatos/reproducoes/_delta_log/00000000000000000000.crc
  → fatos/reproducoes/_delta_log/00000000000000000000.json
  → fatos/reproducoes/ano_mes=__HIVE_DEFAULT_PARTITION__/.part-00000-9b2c4369-f18e-4a5f-beb7-1256d207bb05.c000.snappy.parquet.crc
  → fatos/reproducoes/ano_mes=__HIVE_DEFAULT_PARTITION__/.part-00001-b2f2e06c-9501-46ff-9b44-e6e87c63a6f3.c000.snappy.parquet.crc
  → fatos/reproducoes/ano_mes=__HIVE_DEFAULT_PARTITION__/part-00000-9b2c4369-f18e-4a5f-beb7-1256d207bb05.c000.snappy.parquet
  → fatos/reproducoes/ano_mes=__HIVE_DEFAULT_PARTITION__/part-00001-b2f2e06c-9501-46ff-9b44-e6e87c63a6f3.c000.snappy.parquet
reproducoes Silver concluído — 56,009 plays válidos.


In [7]:
print("\n" + "=" * 55)
print("Silver: pagamentos")
print("=" * 55)

local_bronze = Path("/tmp/bronze_local/pagamentos")
download_delta_do_minio("bronze", "fatos/pagamentos", cliente_minio, local_bronze)
df = spark.read.format("delta").load(str(local_bronze))
print(f"Bronze lido: {df.count():,} registros")

df = (
    df.dropDuplicates(["id"])
      .withColumn("status", F.lower(F.trim(F.col("status"))))
      .withColumn("ano_mes", F.date_format(F.col("data").cast("timestamp"), "yyyy-MM"))
)

local_silver = Path("/tmp/silver/pagamentos")
if local_silver.exists():
    shutil.rmtree(local_silver)

df.write.format("delta").mode("overwrite").partitionBy("ano_mes").save(str(local_silver))
print(f"Silver Delta escrito em {local_silver}")

upload_delta_para_minio(local_silver, "silver", "fatos/pagamentos", cliente_minio)
print(f"pagamentos Silver concluído — {df.count():,} registros.")



Silver: pagamentos
  6 arquivo(s) em bronze/fatos/pagamentos/
Bronze lido: 10,500 registros


Silver Delta escrito em /tmp/silver/pagamentos
  → fatos/pagamentos/_delta_log/.00000000000000000000.crc.crc
  → fatos/pagamentos/_delta_log/.00000000000000000000.json.crc
  → fatos/pagamentos/_delta_log/00000000000000000000.crc
  → fatos/pagamentos/_delta_log/00000000000000000000.json
  → fatos/pagamentos/ano_mes=2023-06/.part-00000-40a6723b-86b0-4f24-a69f-080b9f16c79f.c000.snappy.parquet.crc
  → fatos/pagamentos/ano_mes=2023-06/part-00000-40a6723b-86b0-4f24-a69f-080b9f16c79f.c000.snappy.parquet
  → fatos/pagamentos/ano_mes=2023-07/.part-00000-63d0fb3e-2ce5-46bf-8c9c-002b2ba408bc.c000.snappy.parquet.crc
  → fatos/pagamentos/ano_mes=2023-07/part-00000-63d0fb3e-2ce5-46bf-8c9c-002b2ba408bc.c000.snappy.parquet
  → fatos/pagamentos/ano_mes=2023-08/.part-00000-e2a8f460-e970-4613-828c-7eeb89d8e462.c000.snappy.parquet.crc
  → fatos/pagamentos/ano_mes=2023-08/part-00000-e2a8f460-e970-4613-828c-7eeb89d8e462.c000.snappy.parquet
  → fatos/pagamentos/ano_mes=2023-09/.part-00000-c378ee20-4143-4781-

In [8]:
print("\n" + "=" * 55)
print("Silver: assinaturas")
print("=" * 55)

local_bronze = Path("/tmp/bronze_local/assinaturas")
download_delta_do_minio("bronze", "fatos/assinaturas", cliente_minio, local_bronze)
df = spark.read.format("delta").load(str(local_bronze))
print(f"Bronze lido: {df.count():,} registros")

df = (
    df.dropDuplicates(["id"])
      .withColumn("status", F.lower(F.trim(F.col("status"))))
      .withColumn("ano_mes", F.date_format(F.col("data_inicio").cast("timestamp"), "yyyy-MM"))
)

local_silver = Path("/tmp/silver/assinaturas")
if local_silver.exists():
    shutil.rmtree(local_silver)

df.write.format("delta").mode("overwrite").partitionBy("ano_mes").save(str(local_silver))
print(f"Silver Delta escrito em {local_silver}")

upload_delta_para_minio(local_silver, "silver", "fatos/assinaturas", cliente_minio)
print(f"assinaturas Silver concluído — {df.count():,} registros.")
print("\nTransformação Silver de todas as tabelas fato concluída.")



Silver: assinaturas
  6 arquivo(s) em bronze/fatos/assinaturas/
Bronze lido: 3,500 registros
Silver Delta escrito em /tmp/silver/assinaturas
  → fatos/assinaturas/_delta_log/.00000000000000000000.crc.crc
  → fatos/assinaturas/_delta_log/.00000000000000000000.json.crc
  → fatos/assinaturas/_delta_log/00000000000000000000.crc
  → fatos/assinaturas/_delta_log/00000000000000000000.json
  → fatos/assinaturas/ano_mes=2023-06/.part-00000-f1cbd96c-7a1b-4fab-8d9b-9324c1ef7c73.c000.snappy.parquet.crc
  → fatos/assinaturas/ano_mes=2023-06/part-00000-f1cbd96c-7a1b-4fab-8d9b-9324c1ef7c73.c000.snappy.parquet
  → fatos/assinaturas/ano_mes=2023-07/.part-00000-dcb0020e-2686-49f3-8baf-b5b1cf8d4303.c000.snappy.parquet.crc
  → fatos/assinaturas/ano_mes=2023-07/part-00000-dcb0020e-2686-49f3-8baf-b5b1cf8d4303.c000.snappy.parquet
  → fatos/assinaturas/ano_mes=2023-08/.part-00000-96223c8e-8748-47d0-9176-60085ad58858.c000.snappy.parquet.crc
  → fatos/assinaturas/ano_mes=2023-08/part-00000-96223c8e-8748-47d0-9

In [9]:
spark.stop()
